# Guide-assignment correction

Corrects the crispat guide assignment and writes a `*_gex_guide_corrected.h5ad` per lane, carrying the
corrected `assigned_guide_id`, the original, a `correction` provenance column, and the reannotated target
gene (`assigned_gene_name` / `assigned_gene_id`). Three steps, in order:

**Step 1 — frameshift-twin multiplet → singlet.** A `multi_sgRNA` cell is a cross-hybridization artifact
when its **only** crispat-positive guides are a **frameshift-twin pair** and the **minor** twin carries
`< TWIN_RATIO` (0.30) the UMIs of the **major** twin. Reassign those to a singlet of the major twin.
(Uses crispat per-cell positivity: `guide_batch/{donor}_{cond}_batch_*/guide_assigned.csv`.)

**Step 2 — sub-threshold rescue of unassigned cells.** An **unassigned** cell carrying **exactly one** guide
at **UMI ≥ 3** (no other guide ≥ 3) is a candidate singlet that crispat left below threshold. The rescue is
only trusted for guides passing a **target-activation gate**, learned per donor × condition by pooling
**every downloaded lane**: for each guide with ≥ 10 assigned and ≥ 10 rescue cells, compare its **target
gene** log-norm expression across assigned / rescue / NTC cells. A guide is **rescuable** when

1. its **rescue** cells show **significant activation over NTC** (Welch t-test, BH-FDR < 0.05), **and**
2. the rescue effect size recovers the assigned one: **`d_rescue / d_assigned ≥ 0.8`** (Cohen's d vs NTC).

Cells whose sole ≥ 3 guide is a rescuable guide are reassigned to that guide.

**Step 3 — reannotated target-gene assignment.** Map each cell's corrected `assigned_guide_id` to the target
gene from the `5_sgRNA_annotation` library (`final_gene_name` / `final_gene_id`), so every reannotation fix
(locus reassignments, symbol/gene-id updates) is applied at the cell level. NTC → `NO-TARGET`;
multiplet/unassigned carried through; excluded/unresolved guides → empty.

**Scope.** The notebook runs over **`DONORS` × `CONDITIONS`** (e.g. donor groups `CRaD1/3/4/5` ×
`Rest/Stim8hr/Stim48hr`). Each donor × condition pair is processed **independently** — its own rescuability
gate, its own corrected h5ads — so you can run everything at once *or* one condition (or one donor) at a
time and get identical results. Anything with no downloaded lanes is skipped with a message, so it is safe
to list the full matrix and let the notebook process whatever exists.

**Structure.** *Part 1* builds (or loads) the per-donor-per-condition rescuability gate, pooling every ready
lane, cached to CSV. *Part 2* applies the three steps per lane and writes the corrected h5ad. Lanes are
auto-detected; `WRITE_LANES` can restrict which lanes are *written* (for disk-constrained batching) while
Part 1 still learns the gate from **all** ready lanes.

## Prerequisites & how to run

**Environment.** Python 3 (tested on base conda / miniconda) with `numpy`, `pandas`, `h5py`, `anndata`,
`scipy`, `matplotlib`. No GPU. Part 1 streams every lane's GEX once (~1–2 min/lane warm, more when cold), so
a full 24-lane condition is ~20–40 min; the gate is then cached, so Part 2 and re-runs are fast.

**Required inputs.** `{donor}` is a donor group (`CRaD1`, `CRaD3`, `CRaD4`, `CRaD5`, …) and `{cond}` a
condition (`Rest`, `Stim8hr`, `Stim48hr`). Each donor's data lives in its own folder, resolved by
`data_dir(donor)` — see the path knobs below.

1. **Per-lane processed data** — `data_dir(donor)/{donor}_{cond}_{donor}_L##/`:
   - `{donor}_{cond}_gex_guide.h5ad` — GEX log-norm matrix + crispat `obs['assigned_guide_id']`.
   - `{donor}_{cond}_crispr_preprocessed.h5ad` — raw per-cell guide-UMI matrix, **row-aligned 1:1 with
     `gex_guide`**.
   - (`gex_preprocessed.h5ad` is **not** read here — safe to keep online-only / offloaded.)
2. **crispat per-cell positivity** (Step 1) —
   `data_dir(donor)/guide_batch/{donor}_{cond}_batch_*/guide_assigned.csv` with columns
   `cell, gRNA, UMI_counts`. **Every** batch directory must contain a non-empty CSV, otherwise that donor ×
   condition is treated as an incomplete download and **no output is written** (a partial positive set would
   silently corrupt the twin step). The batch count is auto-detected, so it need not be 16.
3. **sgRNA annotation** — three parquets in `ANNDIR`, produced by the **`5_sgRNA_annotation` pipeline in this
   repo** (they are *not* part of the raw sequencing data — run that pipeline first if missing):
   `CRISPRa_targeting_sgRNA_annotated.parquet` (guide → `final_gene_name`/`final_gene_id`/`is_targeting`;
   supplies the Step-3 target genes), `CRISPRa_NO-TARGET_sgRNA.parquet` (NTC list),
   `CRISPRa_frameshift_twin_pairs.parquet` (twin `guide_a`/`guide_b`). The sgRNA library is shared across
   donor groups, so these are built once and reused.

**Expected layout** (relative paths assume this notebook stays at
`3_codes/PerturbSeq_Analysis_pipeline/src/2_guide-assignment/`):
```
<repo>/
├── 2_data/                                       <- DATA_ROOT
│   └── {donor}_processed/                        <- data_dir(donor); override via DATA_DIRS
│       ├── {donor}_{cond}_{donor}_L##/{donor}_{cond}_{gex_guide,crispr_preprocessed}.h5ad
│       │                                         (+ {donor}_{cond}_gex_guide_corrected.h5ad  <- output)
│       └── guide_batch/{donor}_{cond}_batch_*/guide_assigned.csv
└── 3_codes/PerturbSeq_Analysis_pipeline/src/
    ├── 2_guide-assignment/   (this notebook; writes tables/figures to ./results/)
    └── 5_sgRNA_annotation/results/   (the three annotation parquets)
```
Lanes are **auto-detected** from the `{donor}_{cond}_{donor}_L*` directories, and a lane is used only when
both its h5ad files are locally materialised (`MIN_GEX_BYTES` / `MIN_CRISPR_BYTES`). This makes the notebook
safe to re-run as more lanes finish downloading — it picks up whatever is ready. The helper cell prints, for
every donor in scope, the resolved folder plus the ready-lane and crispat-batch counts, so you can confirm
the paths before anything heavy runs.

**How to run.** Execute all cells top to bottom. Everything is configured in the params cell:

| knob | what it does |
|---|---|
| `DONORS`, `CONDITIONS` | the donor × condition matrix to process. Run **everything available** with `DONORS = ALL_DONORS; CONDITIONS = ALL_CONDITIONS`, or **one at a time** with e.g. `CONDITIONS = ['Rest']`. Pairs with no ready lanes are skipped, so listing the full matrix is safe. |
| `DATA_ROOT`, `DATA_TEMPLATE`, `DATA_DIRS` | where each donor's data lives. Default is `<DATA_ROOT>/<donor>_processed`; put any donor whose folder is named differently in `DATA_DIRS` (e.g. `{'CRaD1': '../../../../2_data/CRaD1_CRI_CAT'}`), which always wins. |
| `WRITE_LANES` | `{(donor, cond): [lanes]}` restricting which lanes get a corrected h5ad written. `{}` = all ready lanes. Part 1's gate **always** pools every ready lane regardless, so a subset write still uses the full-condition rescuable set. Use for disk-constrained batching. |
| `REUSE_RESCUE_TABLE` | reuse the cached `results/{donor}_{cond}_guide_rescuability.csv` instead of recomputing the (slow) gate. **Keep `True` when writing in batches** — see the warning below. |
| `CORRECTED_WITH_X` | `False` = obs-only overlay (~50 MB/lane; join back to `gex_guide` by cell order for expression). `True` = self-contained AnnData carrying GEX `.X` + `counts` layer (~12–16 GB/lane) for direct downstream use. |

> **Batching warning.** If you write a condition in batches (e.g. L01–12, then L13–24 after offloading the
> first half), the gate must be **built once while all lanes are present** and then **reused from cache** for
> the later batches. If the cache were deleted and rebuilt after some lanes were offloaded, it would only see
> the still-local lanes and produce a *different* rescuable set, making the batches inconsistent.

**Outputs.**
- `results/{donor}_{cond}_guide_rescuability.csv` — the per-guide gate (effect sizes, FDRs, `rescuable`).
- `data_dir(donor)/{donor}_{cond}_{lane}/{donor}_{cond}_gex_guide_corrected.h5ad` — obs columns
  `assigned_guide_id`, `assigned_guide_id_original`, `correction` ∈ {`''`, `twin_multiplet_to_singlet`,
  `subthreshold_rescue`}, `assigned_gene_name`, `assigned_gene_id` (NTC → `NO-TARGET`); plus GEX `.X` when
  `CORRECTED_WITH_X=True`.
- `results/guide_assignment_correction_summary.csv` — per-lane counts. **Rows accumulate across runs**
  (matching donor/condition/lane rows are replaced), so batched or partial runs do not lose earlier results.
- `results/guide_rescuability_gate_scatter.png` — one gate panel per donor × condition processed.

> **Note.** If the data lives on a cloud-synced folder (Dropbox/OneDrive "online-only"), files can dehydrate
> to 0 bytes when idle. The readiness check skips such lanes rather than failing, but a file evicted *during*
> a run can still break a read — run `jupyter nbconvert --execute --allow-errors` if you want the executed
> notebook saved regardless.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
import anndata as ad
from scipy.sparse import csr_matrix
from scipy import stats
import matplotlib.pyplot as plt

try:                                    # newer anndata refuses to write nullable-string obs (e.g. _index)
    ad.settings.allow_write_nullable_strings = True
except Exception:
    pass

# ============================ RUN SCOPE ============================
# Everything is processed per (donor, condition) pair, INDEPENDENTLY: each pair gets its own rescuability
# gate and its own corrected h5ads. Listing the whole matrix is safe -- pairs with no downloaded lanes are
# skipped with a message -- so the same notebook covers both usage modes:
#
#   run everything that is available :  DONORS = ALL_DONORS      ; CONDITIONS = ALL_CONDITIONS
#   run a single condition          :  CONDITIONS = ['Rest']
#   run a single donor group        :  DONORS = ['CRaD5']
#
# Results are identical either way (no cross-condition or cross-donor pooling anywhere).
ALL_DONORS     = ['CRaD1', 'CRaD3', 'CRaD4', 'CRaD5']
ALL_CONDITIONS = ['Rest', 'Stim8hr', 'Stim48hr']

DONORS     = ['CRaD5']            # <- set to ALL_DONORS to sweep every donor group
CONDITIONS = ALL_CONDITIONS       # <- e.g. ['Rest'] for a single condition

# ============================ PARAMETERS ============================
# --- Step 2: sub-threshold rescue + target-activation gate ---
UMI_THRESH   = 3       # a guide is "present" in a cell at >= this many UMI
MIN_ASSIGNED = 10      # per-guide cell floor to TEST rescuability (t-test power); no upper cap
MIN_RESCUE   = 10
RESCUE_RATIO = 0.8     # (2) guide rescuable when d_rescue / d_assigned >= this
FDR_ALPHA    = 0.05    # (1) rescue significant vs NTC when BH-FDR < this

# --- Step 1: frameshift-twin multiplet -> singlet ---
TWIN_RATIO = 0.30      # reassign twin-doublet multiplet when minor/major twin UMI < this

# --- lane readiness (guards against cloud-sync "online-only" / partially downloaded files) ---
MIN_GEX_BYTES    = 1e9   # gex_guide.h5ad is many GB when materialised
MIN_CRISPR_BYTES = 1e6   # crispr_preprocessed.h5ad is tens of MB

# --- which lanes get a corrected h5ad WRITTEN ---
# {} -> every ready lane of every (donor, condition) being processed.
# {(donor, cond): [lanes]} -> restrict the write to those lanes (for disk-constrained batching).
# Part 1's gate ALWAYS pools every ready lane regardless of this, so a subset write still uses the
# full-condition rescuable set. Example of a two-batch write:
#   WRITE_LANES = {('CRaD5', 'Stim8hr'): [f'CRaD5_L{i:02d}' for i in range(1, 13)]}   # then 13..25
WRITE_LANES = {}

# Reuse the cached per-(donor, condition) gate CSV instead of recomputing it (the slow step).
# KEEP True when writing a condition in batches: the gate must be built once while every lane is present
# and then reused, otherwise a rebuild after offloading some lanes would give a different rescuable set.
REUSE_RESCUE_TABLE = True

# Corrected-h5ad format:
#   False -> obs-only overlay (~50 MB/lane): corrected assignment columns, NO .X. Join back to gex_guide
#            (identical cell order) when expression is needed. Cheap; good for testing / tight disks.
#   True  -> self-contained: the FULL gex_guide (GEX .X + counts layer + var) with the corrected obs
#            (~12-16 GB/lane). Needs disk headroom + RAM to hold one lane's matrix.
CORRECTED_WITH_X = False

# ============================ PATHS ============================
# Each donor group has its own processed-data folder holding the per-lane directories and guide_batch/.
# The default follows the CRaD5 convention, '<DATA_ROOT>/<donor>_processed'. Donor folders that do not
# follow it (older batches can be named differently) go in DATA_DIRS, which always wins:
#     DATA_DIRS = {'CRaD1': '../../../../2_data/CRaD1_CRI_CAT'}
# Check what a donor resolves to with  data_dir('CRaD1')  after running the helper cell below.
DATA_ROOT     = '../../../../2_data'
DATA_TEMPLATE = '{root}/{donor}_processed'
DATA_DIRS     = {}                                # per-donor overrides, e.g. {'CRaD1': '<path>'}

ANNDIR = '../5_sgRNA_annotation/results'          # outputs of the 5_sgRNA_annotation pipeline
OUT    = 'results'
os.makedirs(OUT, exist_ok=True)

# ============================ sgRNA ANNOTATION (shared across donors) ============================
ntc_ids = set(pd.read_parquet(f'{ANNDIR}/CRISPRa_NO-TARGET_sgRNA.parquet')['guide_id'])
ann = pd.read_parquet(f'{ANNDIR}/CRISPRa_targeting_sgRNA_annotated.parquet')
guide_to_target = dict(zip(ann['guide_id'], ann['final_gene_name']))
tw = pd.read_parquet(f'{ANNDIR}/CRISPRa_frameshift_twin_pairs.parquet')
twinset = {frozenset((a, b)) for a, b in zip(tw['guide_a'], tw['guide_b'])}

# Step 3 maps: corrected guide -> REANNOTATED target gene (name + Ensembl id). The annotated table holds
# every guide (is_targeting flags the NTC block). Fold in NTC -> 'NO-TARGET', targeting-but-no-final-gene
# (excluded / unresolved) -> '', and the assignment sentinels ('', 'nan', 'multi_sgRNA').
guide_to_gene, guide_to_gene_id = {}, {}
for _gid, _nm, _gi, _targ in zip(ann['guide_id'], ann['final_gene_name'], ann['final_gene_id'], ann['is_targeting']):
    if not _targ:
        guide_to_gene[_gid], guide_to_gene_id[_gid] = 'NO-TARGET', ''
    elif pd.isna(_nm):
        guide_to_gene[_gid], guide_to_gene_id[_gid] = '', ''            # excluded / unresolved targeting guide
    else:
        guide_to_gene[_gid] = str(_nm)
        guide_to_gene_id[_gid] = '' if pd.isna(_gi) else str(_gi)       # empty for retired identifiers
for _sp, _g in [('', ''), ('nan', ''), ('multi_sgRNA', 'multi_sgRNA')]:
    guide_to_gene[_sp], guide_to_gene_id[_sp] = _g, ''
guide_to_designed = {g: ('' if pd.isna(v) else str(v))                  # for the "reannotation moved target" stat
                     for g, v in zip(ann['guide_id'], ann['designed_gene_name'])}

# ============================ PLOTTING ============================
COND_COLORS = {'Rest': '#2a78d6', 'Stim8hr': '#eb6834', 'Stim48hr': '#1baf7a'}
BLUE, GREY, ORANGE, GREEN, INK = '#2a78d6', '#8a8f98', '#eb6834', '#1baf7a', '#0b0b0b'
plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.color': '#ececea', 'axes.axisbelow': True})

print(f'scope: donors {DONORS} x conditions {CONDITIONS}')
print(f'annotation: {len(ntc_ids):,} NTC | {int(ann["is_targeting"].sum()):,} targeting guides | '
      f'{len(twinset):,} frameshift-twin pairs')
print(f'rescue: UMI>={UMI_THRESH}, gate = rescue-sig(FDR<{FDR_ALPHA}) & d_rescue/d_assigned>={RESCUE_RATIO} | '
      f'twin ratio<{TWIN_RATIO} | corrected .X: {CORRECTED_WITH_X}')

## Part 1 — rescuability gate, per donor × condition

For each pair in scope, pool **every ready lane** and test **every** guide with ≥ `MIN_ASSIGNED` assigned and
≥ `MIN_RESCUE` rescue cells (no cap on the number of guides). `crispr_preprocessed.h5ad` supplies per-cell
guide UMIs (how many guides clear `UMI_THRESH`, and which is top); `gex_guide.h5ad` supplies the crispat
assignment and the log-norm GEX used to measure target activation.

A guide is **rescuable** when its rescue cells activate the target significantly vs NTC (BH-FDR <
`FDR_ALPHA`) **and** `d_rescue / d_assigned ≥ RESCUE_RATIO`.

The result is cached to `results/{donor}_{cond}_guide_rescuability.csv`. Because the gate pools all ready
lanes independently of `WRITE_LANES`, writing a condition in batches still applies one consistent rescuable
set — provided the cache is reused rather than rebuilt (see the batching warning above).

In [ ]:
from collections import Counter

# ---------------------------------------------------------------- path helpers (donor-aware)
def data_dir(donor):
    """Processed-data folder for a donor group: DATA_DIRS override, else '<DATA_ROOT>/<donor>_processed'.
    Holds the per-lane directories and guide_batch/."""
    return DATA_DIRS.get(donor) or DATA_TEMPLATE.format(root=DATA_ROOT, donor=donor)


def lane_dir(donor, cond, lane):
    return f'{data_dir(donor)}/{donor}_{cond}_{lane}'


def gex_guide_path(donor, cond, lane):
    return f'{lane_dir(donor, cond, lane)}/{donor}_{cond}_gex_guide.h5ad'


def crispr_path(donor, cond, lane):
    return f'{lane_dir(donor, cond, lane)}/{donor}_{cond}_crispr_preprocessed.h5ad'


def corrected_path(donor, cond, lane):
    return f'{lane_dir(donor, cond, lane)}/{donor}_{cond}_gex_guide_corrected.h5ad'


def guide_batch_glob(donor, cond):
    return f'{data_dir(donor)}/guide_batch/{donor}_{cond}_batch_*'


def gate_path(donor, cond):
    return f'{OUT}/{donor}_{cond}_guide_rescuability.csv'


def lane_ready(donor, cond, lane):
    """Both needed h5ad present and locally materialised (not a cloud-sync placeholder)."""
    gg, cp = gex_guide_path(donor, cond, lane), crispr_path(donor, cond, lane)
    return (os.path.exists(gg) and os.path.getsize(gg) > MIN_GEX_BYTES and
            os.path.exists(cp) and os.path.getsize(cp) > MIN_CRISPR_BYTES)


def ready_lanes(donor, cond):
    """Auto-detect fully-downloaded lanes for this donor x condition (any lane count)."""
    prefix = f'{donor}_{cond}_'
    out = []
    for d in glob.glob(f'{data_dir(donor)}/{prefix}{donor}_L*'):
        if not os.path.isdir(d):
            continue
        lane = os.path.basename(d)[len(prefix):]          # -> '{donor}_L##'
        if lane_ready(donor, cond, lane):
            out.append(lane)
    return sorted(out)


def crispat_status(donor, cond):
    """(n_batch_dirs, n_non_empty). crispat splits guides into batches; the batch COUNT is auto-detected
    (need not be 16), and a condition counts as complete only when every batch dir has a non-empty CSV --
    a partial positive set would silently corrupt the Step-1 twin call."""
    dirs = sorted(glob.glob(guide_batch_glob(donor, cond)))
    ok = sum(1 for d in dirs
             if os.path.exists(os.path.join(d, 'guide_assigned.csv'))
             and os.path.getsize(os.path.join(d, 'guide_assigned.csv')) > 0)
    return len(dirs), ok


# ---------------------------------------------------------------- readers
def read_assignment(donor, cond, lane):
    """crispat assigned_guide_id per cell (h5py codes+categories; aligned to crispr cell order)."""
    g = h5py.File(gex_guide_path(donor, cond, lane), 'r')
    cats = np.array([x.decode() if isinstance(x, bytes) else str(x)
                     for x in g['obs/assigned_guide_id/categories'][:]])
    codes = g['obs/assigned_guide_id/codes'][:]
    return np.where(codes >= 0, cats[np.clip(codes, 0, None)], 'nan')


def read_guide_umi(donor, cond, lane):
    """Per cell: (guide names, n guides with UMI>=UMI_THRESH, top such guide's column index).
    crispr_preprocessed X rows are 1:1 aligned (positionally) with gex_guide cells."""
    h = h5py.File(crispr_path(donor, cond, lane), 'r')
    vi = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in h['var/_index/values'][:]])
    shape = tuple(int(s) for s in h['X'].attrs['shape'])
    X = csr_matrix((h['X/data'][:], h['X/indices'][:], h['X/indptr'][:]), shape=shape)
    ge = X.copy(); ge.data = (ge.data >= UMI_THRESH).astype(np.int8)
    n_ge = np.asarray(ge.sum(1)).ravel()
    Xb = X.copy(); Xb.data = (Xb.data >= UMI_THRESH) * X.data
    top = np.asarray(Xb.argmax(1)).ravel()
    return vi, n_ge, top


# ---------------------------------------------------------------- statistics
def bh_fdr(p):
    """Benjamini-Hochberg FDR, NaN-safe: NaN p-values (silent-target guides where a Welch test is
    undefined) pass through as NaN and are excluded rather than poisoning the whole array."""
    p = np.asarray(p, float); q = np.full(p.shape, np.nan); ok = ~np.isnan(p); n = int(ok.sum())
    if n == 0:
        return q
    pv = p[ok]; order = np.argsort(pv); qq = np.empty(n)
    qq[order] = np.minimum.accumulate((pv[order] * n / np.arange(1, n + 1))[::-1])[::-1]
    q[ok] = np.minimum(qq, 1)
    return q


def cohen_d(x, ntc):
    """Effect size vs NTC with average-variance pooled SD (finite even when NTC variance ~0)."""
    sd = np.sqrt((np.var(x, ddof=1) + np.var(ntc, ddof=1)) / 2)
    return (x.mean() - ntc.mean()) / sd if sd > 0 else 0.0


# ---------------------------------------------------------------- Part 1: the rescuability gate
def build_rescuability(donor, cond, lanes):
    """Pool `lanes` of one donor x condition, test every guide with >=MIN_ASSIGNED assigned and
    >=MIN_RESCUE rescue cells for target activation, and flag the rescuable ones.
    Returns + caches a per-guide DataFrame."""
    ntc_arr = np.array(list(ntc_ids))
    lane_arrays = {}
    assigned_ct, rescue_ct = Counter(), Counter()
    guide_names = None
    for lane in lanes:                                    # pool per-cell groups & per-guide counts
        agv = read_assignment(donor, cond, lane)
        vi, n_ge, top = read_guide_umi(donor, cond, lane)
        guide_names = vi if guide_names is None else guide_names
        lane_arrays[lane] = (agv, n_ge, top)
        gene_assigned = np.isin(agv, vi) & ~np.isin(agv, ntc_arr) & (agv != 'multi_sgRNA')
        for g, c in Counter(agv[gene_assigned]).items():
            assigned_ct[g] += c
        resc = (agv == 'nan') & (n_ge == 1)               # unassigned, exactly one guide >= threshold
        for g, c in Counter(vi[top[resc]]).items():
            rescue_ct[g] += c

    cand = pd.concat([pd.Series(assigned_ct, name='assigned'), pd.Series(rescue_ct, name='rescue')],
                     axis=1).fillna(0).astype(int)
    cand = cand[~cand.index.isin(ntc_ids)]
    cand = cand[(cand['assigned'] >= MIN_ASSIGNED) & (cand['rescue'] >= MIN_RESCUE)].copy()  # ALL, no cap
    cand['target_gene'] = cand.index.map(guide_to_target)

    gg = h5py.File(gex_guide_path(donor, cond, lanes[0]), 'r')
    gex_genes = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in gg['var/_index/values'][:]])
    gene2col = {gn: i for i, gn in enumerate(gex_genes)}
    cand = cand[cand['target_gene'].isin(gene2col)].copy()
    target_genes = sorted(set(cand['target_gene']))
    tgene_j = {t: j for j, t in enumerate(target_genes)}
    tcols = np.array([gene2col[t] for t in target_genes])
    remap = np.full(len(gex_genes), -1, int); remap[tcols] = np.arange(len(tcols))
    guide_crispr_col = {g: int(np.where(guide_names == g)[0][0]) for g in cand.index}
    col2guide = {c: g for g, c in guide_crispr_col.items()}
    cand_guides_arr = np.array(list(cand.index)); cand_cols_arr = np.array(list(col2guide))
    print(f'  candidate guides (no cap): {len(cand):,} | target genes: {len(target_genes):,}')

    def extract(lane, coi):
        """(n_coi x n_target) log-norm expr, materialising ONLY cells-of-interest rows -- bounds memory
        to n_coi x n_target instead of n_cells x n_genes."""
        g = h5py.File(gex_guide_path(donor, cond, lane), 'r')
        indptr = g['X/indptr'][:]; n = len(indptr) - 1; data, ind = g['X/data'], g['X/indices']
        idx = np.where(coi)[0]; row2out = np.full(n, -1, int); row2out[idx] = np.arange(len(idx))
        out = np.zeros((len(idx), len(tcols)), np.float32)
        for a in range(0, n, 25000):
            b = min(a + 25000, n); s, e = int(indptr[a]), int(indptr[b]); ii = ind[s:e]
            rows = np.repeat(np.arange(a, b), np.diff(indptr[a:b + 1]))
            keep = (remap[ii] >= 0) & (row2out[rows] >= 0)
            if keep.any():
                out[row2out[rows[keep]], remap[ii[keep]]] = data[s:e][keep]
        return out, row2out

    ntc_expr = []
    assigned_vals = {g: [] for g in cand.index}
    rescue_vals = {g: [] for g in cand.index}
    for lane in lanes:
        agv, n_ge, top = lane_arrays[lane]
        ntc_mask = np.isin(agv, ntc_arr)
        a_mask = np.isin(agv, cand_guides_arr)
        resc_base = (agv == 'nan') & (n_ge == 1) & np.isin(top, cand_cols_arr)
        coi = ntc_mask | a_mask | resc_base
        out, row2out = extract(lane, coi)
        ntc_expr.append(out[row2out[np.where(ntc_mask)[0]]])
        a_rows = np.where(a_mask)[0]
        for g, sub in pd.DataFrame({'row': a_rows, 'g': agv[a_rows]}).groupby('g', sort=False):
            j = tgene_j[cand.loc[g, 'target_gene']]
            assigned_vals[g].append(out[row2out[sub['row'].to_numpy()], j])
        r_rows = np.where(resc_base)[0]
        r_guides = np.array([col2guide[c] for c in top[r_rows]])
        for g, sub in pd.DataFrame({'row': r_rows, 'g': r_guides}).groupby('g', sort=False):
            j = tgene_j[cand.loc[g, 'target_gene']]
            rescue_vals[g].append(out[row2out[sub['row'].to_numpy()], j])
        print(f'  {lane} GEX', end='')
    print()

    ntc_mat = np.vstack(ntc_expr)
    assigned_vals = {g: (np.concatenate(v) if v else np.array([])) for g, v in assigned_vals.items()}
    rescue_vals = {g: (np.concatenate(v) if v else np.array([])) for g, v in rescue_vals.items()}

    rows = []
    for g in cand.index:
        j = tgene_j[cand.loc[g, 'target_gene']]
        ntc = ntc_mat[:, j]; av = assigned_vals[g]; rv = rescue_vals[g]
        rows.append(dict(
            guide=g, target=cand.loc[g, 'target_gene'], n_assigned=len(av), n_rescue=len(rv),
            mean_ntc=float(ntc.mean()), mean_assigned=float(av.mean()), mean_rescue=float(rv.mean()),
            d_assigned=cohen_d(av, ntc), d_rescue=cohen_d(rv, ntc),
            p_assigned_vs_ntc=stats.ttest_ind(av, ntc, equal_var=False, alternative='greater').pvalue,
            p_rescue_vs_ntc=stats.ttest_ind(rv, ntc, equal_var=False, alternative='greater').pvalue))
    res = pd.DataFrame(rows)
    res['ratio_rescue_over_assigned'] = res['d_rescue'] / res['d_assigned'].where(res['d_assigned'] > 0)
    res['fdr_assigned_vs_ntc'] = bh_fdr(res['p_assigned_vs_ntc'])
    res['fdr_rescue_vs_ntc'] = bh_fdr(res['p_rescue_vs_ntc'])
    res['responsive'] = (res['fdr_assigned_vs_ntc'] < FDR_ALPHA) & (res['d_assigned'] > 0)  # assigned activates
    res['rescue_significant'] = (res['fdr_rescue_vs_ntc'] < FDR_ALPHA) & (res['d_rescue'] > 0)   # (1)
    res['rescuable'] = res['rescue_significant'] & (res['ratio_rescue_over_assigned'] >= RESCUE_RATIO)  # (1)&(2)
    res['donor'] = donor; res['condition'] = cond; res['n_lanes_pooled'] = len(lanes)
    res = res.sort_values('d_rescue', ascending=False).reset_index(drop=True)
    res.to_csv(gate_path(donor, cond), index=False)
    return res


# quick sanity print: where does each donor in scope resolve to, and what is available there?
for _d in DONORS:
    _p = data_dir(_d)
    print(f'{_d}: {_p}  {"(exists)" if os.path.isdir(_p) else "(MISSING -- set DATA_DIRS)"}')
    if os.path.isdir(_p):
        for _c in CONDITIONS:
            _n = len(ready_lanes(_d, _c)); _bd, _bo = crispat_status(_d, _c)
            print(f'    {_c:9s} ready lanes {_n:3d} | crispat batches {_bo}/{_bd}')

In [ ]:
# Build (or load) the rescuability gate for every donor x condition in scope.
# The gate ALWAYS pools every ready lane of that pair -- independent of WRITE_LANES -- so a subset write
# still uses the full-condition rescuable set.
rescue_tables = {}
for donor in DONORS:
    for cond in CONDITIONS:
        lanes = ready_lanes(donor, cond)
        if not lanes:
            print(f'[{donor} {cond}] no ready lanes -- skip')
            continue
        path = gate_path(donor, cond)
        if REUSE_RESCUE_TABLE and os.path.exists(path):
            res = pd.read_csv(path)
            print(f'[{donor} {cond}] loaded cached gate ({len(res):,} guides, '
                  f'{res["n_lanes_pooled"].iloc[0]} lanes pooled)  <- not rebuilt')
        else:
            print(f'[{donor} {cond}] building gate over {len(lanes)} ready lanes: {lanes}')
            res = build_rescuability(donor, cond, lanes)
        rescue_tables[(donor, cond)] = res
        print(f'  tested {len(res):,} | responsive {int(res.responsive.sum()):,} | '
              f'rescue-sig(FDR<{FDR_ALPHA}) {int(res.rescue_significant.sum()):,} | '
              f'RESCUABLE(&ratio>={RESCUE_RATIO}) {int(res.rescuable.sum()):,}')

if not rescue_tables:
    print('\nNothing in scope has downloaded lanes -- check DONORS / CONDITIONS / DATA.')
else:
    _d, _c = next(iter(rescue_tables))
    _r = rescue_tables[(_d, _c)]
    print(f'\ntop rescuable guides for {_d} {_c}:')
    display(_r[_r.rescuable].head(10)[['guide', 'target', 'n_assigned', 'n_rescue', 'd_assigned',
                                       'd_rescue', 'ratio_rescue_over_assigned', 'fdr_rescue_vs_ntc']].round(3))

## Part 2 — per-lane correction → `{donor}_{cond}_gex_guide_corrected.h5ad`

For every lane of each donor × condition, apply the three steps to the crispat assignment, in order:

1. **Twin-multiplet → singlet** — a `multi_sgRNA` cell whose crispat positive set is exactly a frameshift-twin
   pair with minor/major UMI < `TWIN_RATIO` → the major twin (`correction = twin_multiplet_to_singlet`).
2. **Sub-threshold rescue** — a still-unassigned cell carrying **exactly one** guide at UMI ≥ `UMI_THRESH`,
   where that guide is **rescuable** per the Part-1 gate → that guide (`correction = subthreshold_rescue`).
3. **Target-gene assignment** — map the corrected `assigned_guide_id` to the **reannotated** target gene
   (`final_gene_name` / `final_gene_id`) → `assigned_gene_name`, `assigned_gene_id`. NTC → `NO-TARGET`;
   `multi_sgRNA` / unassigned carried through; excluded or unresolved targeting guides → empty. This applies
   every reannotation fix (locus reassignments, symbol and gene-id updates) at the cell level.

Output is written **only when crispat is complete** for that donor × condition (every batch directory holds a
non-empty CSV) — a partial positive set would silently corrupt Step 1. Incomplete pairs still print their
provisional counts so you can see where things stand, but nothing is saved.

In [ ]:
def load_positive_sets(donor, cond, lane):
    """crispat per-cell positive guides+UMIs for this lane. Returns {cell: {guide: umi}}."""
    files = sorted(glob.glob(os.path.join(guide_batch_glob(donor, cond), 'guide_assigned.csv')))
    frames = []
    for f in files:
        try:
            d = pd.read_csv(f, usecols=['cell', 'gRNA', 'UMI_counts'])
        except (pd.errors.EmptyDataError, ValueError):
            continue                                   # empty / still-downloading batch
        frames.append(d[d['cell'].str.contains(f'_{lane}_')])
    ga = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['cell', 'gRNA', 'UMI_counts'])
    pos = {}
    for cell, g, u in zip(ga['cell'], ga['gRNA'], ga['UMI_counts']):
        pos.setdefault(cell, {})[g] = u
    return pos


def write_corrected(donor, cond, lane, base_obs, new_cols):
    """Write the corrected h5ad, attaching new_cols (name -> positional array) to obs.
    CORRECTED_WITH_X=False -> obs-only overlay; True -> the full gex_guide (GEX .X + counts layer + var)."""
    outp = corrected_path(donor, cond, lane)
    if CORRECTED_WITH_X:
        adata = ad.read_h5ad(gex_guide_path(donor, cond, lane))     # full: GEX .X + layers + var + obs
        for k, v in new_cols.items():
            adata.obs[k] = v
        adata.write_h5ad(outp)
    else:
        obs = base_obs.copy()
        for k, v in new_cols.items():
            obs[k] = v
        ad.AnnData(obs=obs).write_h5ad(outp)                        # obs-only (empty .X)
    return outp


def correct_lane(donor, cond, lane, rescuable_set, complete, write=True):
    """Twin-multiplet correction + sub-threshold rescue + reannotated target-gene assignment for one lane.
    Writes the corrected h5ad only when `complete` (crispat fully downloaded). Returns a summary dict."""
    obs = ad.read_h5ad(gex_guide_path(donor, cond, lane), backed='r').obs.copy()
    orig = obs['assigned_guide_id'].astype(str)                 # barcode-indexed, in gex_guide order
    new = orig.to_numpy().astype(object).copy()                 # positional, aligned to crispr rows
    corr = np.array([''] * len(new), dtype=object)

    # ---- Step 1: frameshift-twin multiplet -> singlet ----
    pos = load_positive_sets(donor, cond, lane)
    multiplet = set(orig.index[orig == 'multi_sgRNA'])
    twin_rows = []
    for cell in multiplet:
        p = pos.get(cell)
        if p is None or len(p) != 2 or frozenset(p.keys()) not in twinset:
            continue
        (g1, u1), (g2, u2) = p.items()
        (maj, umaj), (mnr, umnr) = ((g1, u1), (g2, u2)) if u1 >= u2 else ((g2, u2), (g1, u1))
        twin_rows.append(dict(cell=cell, major=maj, ratio=umnr / umaj))
    td = pd.DataFrame(twin_rows)
    twin_re = td[td['ratio'] < TWIN_RATIO] if len(td) else td
    pos_of = {b: i for i, b in enumerate(orig.index)}           # barcode -> positional row
    if len(twin_re):
        ti = np.array([pos_of[c] for c in twin_re['cell']])
        new[ti] = twin_re['major'].to_numpy()
        corr[ti] = 'twin_multiplet_to_singlet'

    # ---- Step 2: sub-threshold rescue of unassigned sole-guide cells (gated) ----
    vi, n_ge, top = read_guide_umi(donor, cond, lane)           # positionally aligned to obs rows
    rescuable_cols = np.where(np.isin(vi, list(rescuable_set)))[0]
    top_is_rescuable = np.isin(top, rescuable_cols)             # cell's sole >=thr guide is rescuable
    still_unassigned = np.isin(new, ['nan', ''])                # after Step 1
    rescue_mask = still_unassigned & (n_ge == 1) & top_is_rescuable
    new[rescue_mask] = vi[top[rescue_mask]]
    corr[rescue_mask] = 'subthreshold_rescue'

    # ---- Step 3: reannotated target gene from the corrected guide ----
    gene_name = np.array([guide_to_gene.get(g, '') for g in new], dtype=object)     # NTC -> 'NO-TARGET'
    gene_id = np.array([guide_to_gene_id.get(g, '') for g in new], dtype=object)
    designed = np.array([guide_to_designed.get(g, '') for g in new], dtype=object)
    is_gene_cell = ~np.isin(gene_name, ['', 'NO-TARGET', 'multi_sgRNA'])
    n_gene = int(is_gene_cell.sum())
    n_ntc = int((gene_name == 'NO-TARGET').sum())
    n_gene_reassigned = int((is_gene_cell & (gene_name != designed)).sum())   # reannotation moved the target

    n_twin = int((corr == 'twin_multiplet_to_singlet').sum())
    n_resc = int(rescue_mask.sum())
    tag = 'complete' if complete else 'INCOMPLETE crispat - provisional, NOT written'
    print(f'[{donor} {cond} {lane}] multiplet {len(multiplet):,} | twin->singlet {n_twin:,} | '
          f'rescued {n_resc:,} | multi_sgRNA {len(multiplet):,}->{int((new=="multi_sgRNA").sum()):,} | '
          f"unassigned {int(np.isin(orig.to_numpy(),['nan','']).sum()):,}->{int(np.isin(new,['nan','']).sum()):,} | {tag}")
    print(f'    gene: {n_gene:,} targeting-gene cells | {n_ntc:,} NTC | '
          f'{n_gene_reassigned:,} reannotated to a gene != designed')

    if write and complete:
        cols = {'assigned_guide_id_original': orig.to_numpy(), 'assigned_guide_id': new, 'correction': corr,
                'assigned_gene_name': gene_name, 'assigned_gene_id': gene_id}
        outp = write_corrected(donor, cond, lane, obs, cols)
        print(f'    -> wrote {outp}  ({"with GEX .X" if CORRECTED_WITH_X else "obs-only"})')

    return dict(donor=donor, condition=cond, lane=lane, complete=complete, n_cells=len(new),
                n_multiplet=len(multiplet), n_twin_reassigned=n_twin, n_rescued=n_resc,
                n_unassigned_before=int(np.isin(orig.to_numpy(), ['nan', '']).sum()),
                n_unassigned_after=int(np.isin(new, ['nan', '']).sum()),
                n_gene_cells=n_gene, n_ntc_cells=n_ntc, n_gene_reassigned=n_gene_reassigned,
                written=bool(write and complete))


# ---------------------------------------------------------------- run Part 2
corr_summary = []
for (donor, cond), gate in rescue_tables.items():
    rescuable_set = set(gate.loc[gate['rescuable'], 'guide'])
    n_dirs, n_ok = crispat_status(donor, cond)
    complete = (n_dirs > 0 and n_ok == n_dirs)
    lanes = WRITE_LANES.get((donor, cond), ready_lanes(donor, cond))
    print(f'\n=== [{donor} {cond}] rescuable guides: {len(rescuable_set):,} | '
          f'crispat batches {n_ok}/{n_dirs}{"" if complete else "  <-- INCOMPLETE, no output"} | '
          f'lanes to write: {len(lanes)} ===')
    for lane in lanes:
        if not lane_ready(donor, cond, lane):
            print(f'[{donor} {cond} {lane}] not ready -- skip')
            continue
        corr_summary.append(correct_lane(donor, cond, lane, rescuable_set, complete))

# Persist the per-lane summary. Rows ACCUMULATE across runs (same donor/condition/lane is replaced) so a
# batched or partial run never discards results recorded by an earlier one.
corr_df = pd.DataFrame(corr_summary)
summary_path = f'{OUT}/guide_assignment_correction_summary.csv'
if len(corr_df):
    if os.path.exists(summary_path):
        prev = pd.read_csv(summary_path)
        key = ['donor', 'condition', 'lane']
        if all(k in prev.columns for k in key):
            merged = prev.merge(corr_df[key].assign(_new=1), on=key, how='left')
            prev = merged[merged['_new'].isna()].drop(columns='_new')
            corr_df = pd.concat([prev, corr_df], ignore_index=True)
    corr_df = corr_df.sort_values(['donor', 'condition', 'lane']).reset_index(drop=True)
    corr_df.to_csv(summary_path, index=False)
corr_df

## QC — rescuability gate and corrected-object verification

In [ ]:
# (a) one rescuability-gate panel per donor x condition; (b) verify every corrected h5ad written this run.
if rescue_tables:
    keys = list(rescue_tables)
    ncol = min(3, len(keys))
    nrow = int(np.ceil(len(keys) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow), squeeze=False)
    for ax, (donor, cond) in zip(axes.ravel(), keys):
        r = rescue_tables[(donor, cond)]
        ok = r['rescuable'].values
        hi = float(np.nanmax([r.d_assigned.max(), r.d_rescue.max()])) * 1.05
        ax.scatter(r.d_assigned[~ok], r.d_rescue[~ok], s=16, c=GREY, alpha=0.6,
                   label=f'not rescuable ({int((~ok).sum())})')
        ax.scatter(r.d_assigned[ok], r.d_rescue[ok], s=20, c=COND_COLORS.get(cond, BLUE), alpha=0.9,
                   label=f'rescuable ({int(ok.sum())})')
        ax.plot([0, hi], [0, hi], color=INK, ls='--', lw=1, label='equal')
        ax.plot([0, hi], [0, RESCUE_RATIO * hi], color=ORANGE, ls=':', lw=1.5, label=f'ratio = {RESCUE_RATIO}')
        ax.set_xlim(-0.1, hi); ax.set_ylim(-0.1, hi)
        ax.set_xlabel("assigned activation (Cohen's d vs NTC)")
        ax.set_ylabel("rescue activation (Cohen's d vs NTC)")
        ax.set_title(f"{donor} {cond}\n({r['n_lanes_pooled'].iloc[0]} lanes, {len(r):,} guides tested)", fontsize=10)
        ax.legend(frameon=False, fontsize=8.5, loc='upper left')
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
    for ax in axes.ravel()[len(keys):]:
        ax.set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{OUT}/guide_rescuability_gate_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

# ---- verify each corrected h5ad written in THIS run: provenance + target-gene partition ----
for s in corr_summary:
    if not s['written']:
        continue
    p = corrected_path(s['donor'], s['condition'], s['lane'])
    o = ad.read_h5ad(p, backed='r').obs
    ag = o['assigned_guide_id'].astype(str)
    changed = (ag != o['assigned_guide_id_original'].astype(str))
    print(f"[{s['donor']} {s['condition']} {s['lane']}] {os.path.basename(p)}: {o.shape[0]:,} cells")
    print(f"    correction counts = {o['correction'].value_counts().to_dict()} | total changed = "
          f"{int(changed.sum()):,} (twin {s['n_twin_reassigned']:,} + rescue {s['n_rescued']:,})")
    # these five categories partition every cell -- the sum is asserted below
    gn = o['assigned_gene_name'].astype(str)
    unassigned = ag.isin(['', 'nan'])
    special = gn.isin(['', 'NO-TARGET', 'multi_sgRNA'])
    cats = {'unassigned': int(unassigned.sum()),
            'multi_sgRNA': int((gn == 'multi_sgRNA').sum()),
            'NTC': int((gn == 'NO-TARGET').sum()),
            'targeting_gene': int((~special).sum()),
            'excluded_no_gene': int(((~unassigned) & (gn == '') & (ag != 'multi_sgRNA')).sum())}
    total = sum(cats.values())
    print(f"    genes: {cats} | sum={total:,} (== n_cells: {total == o.shape[0]}) | "
          f"unique target genes = {gn[~special].nunique():,} | reannotated!=designed = {s['n_gene_reassigned']:,}")
    assert {'assigned_gene_name', 'assigned_gene_id'}.issubset(o.columns), 'gene columns missing!'
    assert total == o.shape[0], 'gene categories do not partition all cells!'
    assert int(changed.sum()) == s['n_twin_reassigned'] + s['n_rescued'], 'changed-cell count mismatch!'